# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Maximiliano Romano 
- Alumno 2 : Facundo Molina

In [ ]:
## TO-DO utiliza esta notebook para documentar, entrenar y probar el modelo.

**LIBRERÍAS**

In [ ]:
import cv2
import os
from pathlib import Path
from sklearn.datasets import fetch_lfw_people
import numpy as np
import insightface
from PIL import Image
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from facenet_pytorch import InceptionResnetV1
import torch
from torchvision import transforms
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder

**DATASET**

In [5]:
# DESCARGA DE DATASET LFW, RECORTE DE 20 PRINCIPALES Y GUARDADO EN CARPETA
lfw_people = fetch_lfw_people(min_faces_per_person=40, resize=1.0)

unique, counts = np.unique(lfw_people.target, return_counts=True)

top_20 = unique[np.argsort(counts)[-20:]]

mask = np.isin(lfw_people.target, top_20)
X = lfw_people.images[mask]
y = lfw_people.target[mask]


new_labels = {old: new for new, old in enumerate(top_20)}
y = np.array([new_labels[label] for label in y])

target_names = lfw_people.target_names[top_20]

base_path = Path(r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp1\tuia-face-recognition-app\data\processed")


for name in target_names:
    person_path = base_path / name.replace(" ", "_")
    person_path.mkdir(parents=True, exist_ok=True)

for i, (img, label_idx) in enumerate(zip(X, y)):

    name = target_names[label_idx].replace(" ", "_")

    img_to_save = (img * 255).astype(np.uint8)

    img_to_save = cv2.cvtColor(img_to_save, cv2.COLOR_GRAY2BGR)

    file_name = f"{name}_{i:04d}.jpg"
    file_path = base_path / name / file_name

    cv2.imwrite(str(file_path), img_to_save)

**DETECCIÓN Y ALINEACIÓN**

In [ ]:
# ALINEACIÓN, DETECCIÓN, RECONOCIMIENTO, EMBEDDING CON INSIGHTFACE. MÁS RÁPIDA Y APLICA TODO EN UN PASO

dataset = Path('src/data/processed')
output_dir = Path("output/post-processed")
output_dir.mkdir(parents=True, exist_ok=True)

pil_faces = {}
embeddings_db = {}

for folder in dataset.iterdir():
    if folder.is_dir():
        persona_name = folder.name  
        
        for img_path in folder.glob('*.jpg'): 
            # 1. Load Image
            pil_img = Image.open(img_path).convert('RGB')
            k = f"{persona_name}_{img_path.name}"
            
            img_array = np.array(pil_img)
            img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)

            boxes, _ = service.detect_faces(img_bgr)
            
            if not boxes:
                print(f"No face detected for {k}")
                # Fallback: Resize original
                fallback = pil_img.resize((service.face_size, service.face_size))
                pil_faces[k] = fallback
                fallback.save(output_dir / f"{k}_noface.jpg")
            else:
                for i, box in enumerate(boxes):
                    aligned_obj = service.align_face(img_bgr, box)
                    if aligned_obj is None: 
                        continue
                    
                    
                    face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
                    face_pil = Image.fromarray(face_rgb)
                    
                    face_id = f"{k}_{i}"
                    pil_faces[face_id] = face_pil
                    face_pil.save(output_dir / f"{face_id}.jpg")
                    
                    
                    embedding = service.extract_embedding_from_face(aligned_obj)
                    embeddings_db[face_id] = embedding
                    #service.store.append(...)

    

AttributeError: 'str' object has no attribute 'items'

**ARQUITECTURA(CON INCEPTIONRESNETV1)**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

modelo = InceptionResnetV1(pretrained="vggface2").eval().to(device)

print(f"Parámetros: {sum(p.numel() for p in modelo.parameters()):,}")

dummy = torch.randn(1, 3, 160, 160).to(device)

with torch.no_grad():
    emb = modelo(dummy)
    
print(f"Embedding shape: {emb.shape}  norma L2: {emb.norm().item():.4f} (esperado ~ 1.0)")

**EXTRACCIÓN DE EMBEDDINGS**

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])


for i, box in enumerate(boxes):
    aligned_obj = service.align_face(img_bgr, box)
    if aligned_obj is None: 
        continue

    face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
    face_pil = Image.fromarray(face_rgb)

    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)

    with torch.no_grad():

        embedding_tensor = modelo(face_tensor)

        embedding_flat = embedding_tensor.squeeze().cpu().numpy()

    face_id = f"{k}_{i}"
    embeddings_db[face_id] = embedding_flat

    face_pil.save(output_dir / f"{face_id}.jpg")

**VERIFICACIÓN**

In [ ]:
def verificar(emb1, emb2, umbral=0.6):
    sim = float(np.dot(emb1, emb2))
    return {"similitud": round(sim, 4), "misma_persona": sim >= umbral}

emb_array = np.array(list(embeddings_db.values()))
emb_labels = list(embeddings_db.keys())


sim_matrix = emb_array @ emb_array.T


fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)

plt.colorbar(im, ax=ax, label="Cosine Similarity")


ticks = range(len(emb_labels))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(emb_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(emb_labels, fontsize=8)


for i in range(len(emb_labels)):
    for j in range(len(emb_labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", 
                ha="center", va="center", color="black", fontsize=7)

ax.set_title("Matriz de Similitud entre Imágenes Procesadas")
plt.tight_layout()
plt.show()

**EVALUACIÓN**

**FINE-TUNING**

In [ ]:
own_tensors = []
own_labels_raw = []

face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
face_pil = Image.fromarray(face_rgb)

face_t = preprocess(face_pil) 

own_tensors.append(face_t)
own_labels_raw.append(persona_name)


le_ft = LabelEncoder()
y_all = torch.tensor(le_ft.fit_transform(own_labels_raw), dtype=torch.long)
X_all = torch.stack(own_tensors)

loader = DataLoader(TensorDataset(X_all, y_all), batch_size=8, shuffle=True)
print(f"Fine-tuning: {len(own_tensors)} imágenes | {len(le_ft.classes_)} clases")


for p in modelo.parameters():
    p.requires_grad = False


for p in modelo.last_linear.parameters():
    p.requires_grad = True
for p in modelo.last_bn.parameters():
    p.requires_grad = True

num_clases = len(le_ft.classes_)
head = nn.Linear(512, num_clases).to(device)

optimizer = torch.optim.Adam(
    list(modelo.last_linear.parameters()) + 
    list(modelo.last_bn.parameters()) + 
    list(head.parameters()), 
    lr=1e-4
)
criterion = nn.CrossEntropyLoss()


modelo.train()
head.train()

for epoch in range(10):
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        

        embeddings = modelo(batch_x)

        outputs = head(embeddings)
        
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(loader):.4f}")



**GUARDADO**

In [ ]:
# from src.lib.schemas import AlignedFace, EmbeddingRecord, InsertRequest